In [21]:
import numpy as np
import yfinance as yf
import pandas as pd
import os
import matplotlib.pyplot as plt
import torch
import cvxpy as cp
from torch.utils.data import DataLoader
from dataclasses import dataclass, asdict
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from pykalman import KalmanFilter
import torch_geometric.nn as geom_nn
from torch.utils.data import TensorDataset, DataLoader

In [4]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def get_data(
      self,
      indices:list,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    macros=["^TNX", "^FVX", "DX-Y.NYB", "^TRCCRB"]
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"
    benchmark_path = f"benchmark_{start}_{end}.parquet"
    macros_path = f"macros_{start}_{end}.parquet"

    if not os.path.exists(df_path) or not os.path.exists(benchmark_path):
      df = yf.download(indices, start, end, interval)
      bench_tmp = yf.download([benchmark], start, end, interval)

      bench_data = bench_tmp[["Close"]]
      macro_data = yf.download(macros, start, end, interval)[["Close"]]

      df.to_parquet(df_path)
      bench_data.to_parquet(benchmark_path)
      macro_data.to_parquet(macros_path)

    else:
      df = pd.read_parquet(df_path)
      bench_data = pd.read_parquet(benchmark_path)
      macro_data = pd.read_parquet(macros_path)

    benchmark = bench_data.pct_change().dropna()
    self.universe = df.columns

    return df, benchmark, macro_data

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [5]:
class FeatureGenerator:
  def __init__(self, kalman_params, debug:bool=False, **kwargs):
    super().__init__(
      debug=debug,
      **kwargs
    )
    self.debug = debug
    self.kalman_params = asdict(kalman_params) if isinstance(kalman_params, KalmanParams) else kalman_params

  def garman_klass_vol(self, data):
    hl = np.square(np.log(data["High"]/data["Low"]))
    co = np.square(np.log(data["Close"]/data["Open"]))
    GK = (0.5*hl - (2*np.log(2))*co)

    return GK

  def generate_kalman_outputs(self, log_prices):
    kf_dict = {}
    dt = self.kalman_params["dt"]
    trans_mat = np.array(
      [
        [1, 1, dt],
        [0, 1, 1],
        [0, 0, 1]
      ]
    )
    obs_mat = np.array(
      [
        [1, 0, 0]
      ]
    )

    cwna_mtx = np.array(
      [
        [dt**5/20, dt**4/8, dt**3/6],
        [dt**4/8, dt**3/4, dt**2/2],
        [dt**3/6, dt**2/2, dt]
      ]
    )

    obs_cov = [[1.0]]
    init_cov = np.zeros_like(cwna_mtx)
    np.fill_diagonal(init_cov, [1]+[10]*(len(np.diag(cwna_mtx))-1))

    for col in log_prices.columns:
      init_state = np.array([log_prices[col].iloc[0], 0, 0])

      for i in ["Fast", "Mid", "Slow"]:

        trans_cov = self.kalman_params[i]*cwna_mtx
        col_name = f"{col}{i}"
        kf = KalmanFilter(
          transition_matrices=trans_mat,
          observation_matrices=obs_mat,
          transition_covariance=trans_cov,
          observation_covariance=obs_cov,
          initial_state_mean=init_state,
          initial_state_covariance=init_cov
        )

        state_means, _ = kf.filter(log_prices[col])

        denoised_values = state_means[:, 0]
        momentum_values = state_means[:, 1]
        acceleration_values = state_means[:, 2]

        kf_dict[col_name + "Velocity"] = momentum_values
        kf_dict[col_name + "Acceleration"] = acceleration_values

    return kf_dict

  def get_betas(self, X, y):
    cov_mtx = np.cov(X, y)
    cov = cov_mtx[0, 1]
    var = cov_mtx[0, 0]
    return cov / var

  def get_macro_features(self, macro_data, market_returns, window=21):
    cols = ['spread_change', 'usd_idx_change', 'cmd_idx_change']
    spread = macro_data["^TNX"] - macro_data["^FVX"]
    spread_change = spread.pct_change()
    usd_idx_change = macro_data["DX-Y.NYB"].pct_change()
    cmd_idx_change = macro_data["^TRCCRB"].pct_change()

    macro_df = pd.concat(
      [spread_change, usd_idx_change, cmd_idx_change],
      axis=1
    )
    macro_df.columns = cols

    market_tmp = market_returns.copy()

    market_tmp.columns = ['market']

    combined = pd.concat([macro_df, market_tmp], axis=1).dropna()

    del (market_returns, macro_df, market_tmp, spread, spread_change, usd_idx_change, cmd_idx_change)

    rolling_betas = pd.DataFrame(index=combined.index)

    for col in cols:
      betas = []
      for i in range(len(combined)):
        if i < window - 1:
          betas.append(np.nan)
        else:
          window_data = combined.iloc[i - window + 1 : i + 1]
          X = window_data['market'].values
          y = window_data[col].values
          betas.append(self.get_betas(X, y))

      rolling_betas[f'{col}_beta'] = betas

    return rolling_betas

  def generate_ex_R_features(self, data, window:int=21):
    ex_r_features = {}

    gk_vol = self.garman_klass_vol(data)
    for i in gk_vol.columns:
      ex_r_features["GKVol"+i] = gk_vol[i].values


    kalman_outputs = self.generate_kalman_outputs(np.log(data["Close"]))

    ex_r_features |= kalman_outputs

    ex_R_target = np.log(data["Close"].pct_change()+1).shift(-window).dropna()

    ex_r_features = pd.DataFrame(ex_r_features, index=data.index)

    return ex_r_features, ex_R_target

  def generate_cov_features(self, data, macro_data, market_returns, window=21):
    cov_features = {}

    gk_vol = self.garman_klass_vol(data)
    for i in gk_vol.columns:
      cov_features["GKVol"+i] = gk_vol[i].values

    cov_features = pd.DataFrame(cov_features, index=data.index)
    macro_betas = self.get_macro_features(
        macro_data=macro_data["Close"],
        market_returns=market_returns,
        window=window
    )

    cov_features_full = pd.concat([cov_features, macro_betas], axis=1).dropna()

    cov_target = data["Close"].pct_change()\
                              .rolling(window)\
                              .cov()\
                              .shift(-window)\
                              .dropna()

    return cov_features_full, cov_target

  def generate_features(self, data, macro_data, market_returns, window=21):
    ex_r_features, ex_r_target = self.generate_ex_R_features(data, window)
    cov_features, cov_target = self.generate_cov_features(data, macros, market_returns)

    multi_idx = cov_target.index.get_level_values(0).unique()

    cm_idx = ex_r_features.index\
                          .intersection(cov_features.index)\
                          .intersection(ex_r_target.index)\
                          .intersection(multi_idx)

    cov_target = cov_target.loc[cov_target.index.isin(cm_idx, level=0)]
    ex_r_features = ex_r_features.loc[cm_idx]
    cov_features = cov_features.loc[cm_idx]
    ex_r_target = ex_r_target.loc[cm_idx]

    return ex_r_features, ex_r_target, cov_features, cov_target




In [6]:
data, benchmark, macros = DataStore().get_data(tickers, "2021-01-01", "2026-06-01", interval="1d")

/tmp/ipykernel_1535/131240113.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(indices, start, end, interval)
[*********************100%***********************]  3 of 3 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['KMLM']: OperationalError('database is locked')
/tmp/ipykernel_1535/131240113.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  bench_tmp = yf.download([benchmark], start, end, interval)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_1535/131240113.py:28: FutureWarning: YF.download() has changed argument auto_adjust default to True
  macro_data = yf.download(macros, start, end, interval)[["Close"]]
[*********************100%***********************]  4 of 4 completed


In [22]:
@dataclass
class KalmanParams:
  dt:float = 1.0
  Fast:float = 1e-1
  Mid:float = 1e-3
  Slow:float = 1e-5

@dataclass
class ExRModelParams:
  d_mode:int = 64
  max_len:int = 100
  num_features:int = None# change num features to shape of the the shape of input
  nhead:int = 4
  num_layers:int = 2
  seq_len:int = 21

@dataclass
class CovModelParams:
    num_node_features: int = None
    hidden_dim: int = 32
    dropout: float = 0.1
    num_assets: int = None

In [9]:
fg = FeatureGenerator(
    kalman_params= KalmanParams()
)

In [10]:
ex_r_features, ex_r_target, cov_features, cov_target = fg.generate_features(data, macros, benchmark, window=21)

/tmp/ipykernel_1535/2322507203.py:128: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ex_R_target = np.log(data["Close"].pct_change()+1).shift(-window).dropna()
/tmp/ipykernel_1535/2322507203.py:81: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  spread_change = spread.pct_change()
/tmp/ipykernel_1535/2322507203.py:83: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  cmd_idx_change = macro_data["^TRCCRB

In [11]:
class TemporalPosEncoding(nn.Module):
  def __init__(self, d_model, max_len=100):
    super().__init__()
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    pe = pe.unsqueeze(0).unsqueeze(0)
    self.register_buffer('pe', pe)

  def forward(self, x):
    return x + self.pe[:, :, :x.size(2), :]

In [12]:
class SpatialTemporalTransformer(nn.Module):
  def __init__(self, model_params):
    super().__init__()
    self.feature_embed = nn.Linear(
      model_params.num_features,
      model_params.d_model
    )

    self.pos_encoder = TemporalPosEncoding(
      model_params.d_model,
      model_params.max_len
    )

    self.temporal_encoder = self.transformer(model_params)
    self.spatial_encoder = self.transformer(model_params)

    self.output_head = nn.Linear(
      model_params.d_model,
      1
    )

  def forward(self, x):
    B, N, T, F = x.shape
    x = self.feature_embed(x)
    x = self.pos_encoder(x)
    x = x.permute(2, 0, 1, 3).reshape(T, B * N, -1)
    x = self.temporal_encoder(x)
    x = x[-1, :, :].reshape(B, N, -1)
    x = x.permute(1, 0, 2)
    x = self.spatial_encoder(x)
    x = x.permute(1, 0, 2)
    return self.output_head(x).squeeze(-1)

  def transformer(self, model_params):
    transformer_layer = nn.TransformerEncoderLayer(
      d_model=model_params.d_model,
      nhead=model_params.nhead,
      dim_feedforward=model_params.d_model * 4,
      batch_first=False
    )
    return nn.TransformerEncoder(
      transformer_layer,
      num_layers=model_params.num_layers
    )

In [20]:
class CovarianceGNN(nn.Module):
  def __init__(self, model_params):
      super().__init__()
      self.num_assets = model_params.num_assets

      self.conv1 = geom_nn.GATv2Conv(
        model_params.num_node_features,
        model_params.hidden_dim,
        heads=2,
        dropout=model_params.dropout,
        concat=True
      )

      self.conv2 = geom_nn.GATv2Conv(
        model_params.hidden_dim * 2,
        model_params.hidden_dim,
        heads=1,
        dropout=model_params.dropout,
        concat=False
      )

      half = model_params.hidden_dim // 2

      self.idiosyncratic_head = nn.Sequential(
        nn.Linear(model_params.hidden_dim, half),
        nn.ReLU(),
        nn.Linear(half, 1)
      )

      self.systematic_head = nn.Sequential(
        nn.Linear(model_params.hidden_dim * 2, half),
        nn.ReLU(),
        nn.Linear(half, 1)
      )

  def forward(self, x, edge_index):
    h = torch.relu(self.conv1(x, edge_index))
    h = torch.relu(self.conv2(h, edge_index))

    log_diag = self.idiosyncratic_head(h).squeeze(-1)
    diag_mtx = torch.diag_embed(torch.exp(log_diag))

    L = torch.zeros((self.num_assets, self.num_assets), device=x.device)

    row, col = edge_index
    for i in range(edge_index.size(1)):
      u, v = row[i].item(), col[i].item()
      if u >= v:
        edge_feat = torch.cat([h[u], h[v]], dim=-1)
        L[u, v] = self.systematic_head(edge_feat)

    covariance_mtx = torch.mm(L, L.t()) + diag_mtx
    return covariance_mtx

In [23]:
class Optimizer:
  def __init__(
      self,
      turnover_penalty_weight=0.5,
      risk_aversion=3.0,
      min_weight_threshold=0.01,
      debug:bool=False, **kwargs
    ):

    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug
    self.turnover_penalty = turnover_penalty_weight
    self.risk_aversion = risk_aversion
    self.min_weight_threshold = min_weight_threshold

  def optimize_portfolio(
    self,
    ex_returns,
    fwd_cov,
    w_prev=None
  ):
    S, N = ex_returns.shape
    R = ex_returns.values

    w = cp.Variable(N)
    u = cp.Variable(S)
    zeta = cp.Variable()

    ex_R = ex_returns.values @ w

    constraints = [
        u >= -R @ w - zeta,
        u >= 0,
        cp.sum(w) == 1,
        w >= 0,
    ]

    if w_prev is not None:
        w_prev_arr = np.asarray(w_prev, dtype=float)
        if w_prev_arr.shape[0] != N:
            raise ValueError(
                f"w_prev length {w_prev_arr.shape[0]} != current N={N}. "
                "Caller must align w_prev to the current filtered universe."
            )

        turnover_penalty = self.turnover_penalty_weight * cp.norm1(w - w_prev_arr)
    else:
        turnover_penalty = 0.0

    ptf_risk = cp.sqrt(cp.quad_form(w, fwd_cov))


    obj_fn = cp.Maximize(ex_R - self.risk_aversion * ptf_risk - turnover_penalty)
    problem = cp.Problem(obj_fn, constraints)
    problem.solve(solver=cp.CLARABEL)

    if problem.status not in ["optimal", "optimal_inaccurate"]:
        raise ValueError(f"Optimization failed: {problem.status}")

    return w

  def process_w(self, w):
    weights = np.array(w.value)
    weights[weights < self.min_weight_threshold] = 0.0

    total_w = np.sum(weights)
    if total_w > 0:
        weights_recalc = weights / total_w

    return weights_recalc

In [ ]:
class Portfolio(Optimizer, DataStore, FeatureGenerator):
  def __init__(self, recalc_window, optim_params, returns_gen_params,
                model_dir="models", debug: bool = False, **kwargs):
    super().__init__(
        recalc_window,
        **optim_params,
        **returns_gen_params,
        debug=debug,
        **kwargs
    )
    self.debug = debug
    self.recalc_window = recalc_window
    self.model_dir = model_dir
    os.makedirs(self.model_dir, exist_ok=True)
    self.ex_r_model = None
    self.cov_model = None
    self.edge_index = None

  def convert_to_simple(self, log_returns, prev_returns):
    arithm_returns = np.exp(log_returns) - 1 + 0.5 * (prev_returns.std()) ** 2
    return arithm_returns

  def _asset_tickers(self):
    return (list(self.universe.get_level_values(-1).unique())
            if hasattr(self.universe, "get_level_values")
            else list(self.universe)
          )

  def _stack_ex_r_tensor(self, ex_r_features, seq_len):
    tickers = self._asset_tickers()
    per_asset, feature_names = [], None
    for t in tickers:
      cols = sorted(c for c in ex_r_features.columns if t in c)
      asset_df = ex_r_features[cols]
      if feature_names is None:
          feature_names = cols
      per_asset.append(asset_df.values)

    arr = np.stack(per_asset, axis=1)
    T = arr.shape[0]

    if T < seq_len:
      raise ValueError(f"Not enough history ({T}) for seq_len={seq_len}")

    windows = np.stack(
      [arr[i - seq_len:i] for i in range(seq_len, T + 1)],
      axis=0
    )

    windows = np.transpose(windows, (0, 2, 1, 3))
    return (
        torch.tensor(windows, dtype=torch.float32),
        ex_r_features.index[seq_len - 1:]
    )

  def _fully_connected_edge_index(self, n):
    idx = torch.combinations(torch.arange(n), r=2).t()
    return torch.cat([idx, idx.flip(0)], dim=1)

  def _cov_node_features(self, cov_features):
    tickers = self._asset_tickers()
    gk_cols = [c for c in cov_features.columns if c.startswith("GKVol")]
    market_cols = [c for c in cov_features.columns if c not in gk_cols]
    node_feats = []
    for t in tickers:
      col = f"GKVol{t}"
      gk = cov_features[col] \
                if col in cov_features.columns \
                else pd.Series(0.0, index=cov_features.index)

      feats = pd.concat(
        [gk.rename("gk_vol"), cov_features[market_cols]],
        axis=1
      )
      node_feats.append(feats.values)
    return np.stack(node_feats, axis=1)

  def train_ex_r_model(self, ex_r_model_params, ex_r_features, ex_r_target,
                        epochs=50, lr=1e-3, batch_size=32):
      X, idx = self._stack_ex_r_tensor(ex_r_features, ex_r_model_params.seq_len)
      y_df = ex_r_target.reindex(idx).dropna()
      X = X[:len(y_df)]
      y = torch.tensor(y_df.values, dtype=torch.float32)

      ex_r_model_params.num_features = X.shape[-1]
      model = SpatialTemporalTransformer(ex_r_model_params)
      opt = torch.optim.Adam(model.parameters(), lr=lr)
      loss_fn = nn.MSELoss()

      dl = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
      model.train()
      for epoch in range(epochs):
        epoch_loss = 0.0
        for xb, yb in dl:
          opt.zero_grad()
          loss = loss_fn(model(xb), yb)
          loss.backward()
          opt.step()
          epoch_loss += loss.item() * xb.size(0)
        if self.debug:
          print(f"[ex_r] epoch {epoch+1}/{epochs} loss={epoch_loss/len(X):.6f}")

      self.ex_r_model = model
      torch.save(
          {
            "state_dict": model.state_dict(),
            "params": asdict(ex_r_model_params)
          },
                  os.path.join(self.model_dir, "ex_r_model.pt"))
      return model

  def train_cov_model(
      self,
      cov_model_params,
      cov_features,
      cov_target,
      epochs=50,
      lr=1e-3
    ):
      node_arr = self._cov_node_features(cov_features)
      n_assets = node_arr.shape[1]
      cov_model_params.num_assets = n_assets
      cov_model_params.num_node_features = node_arr.shape[-1]
      edge_index = self._fully_connected_edge_index(n_assets)

      model = CovarianceGNN(cov_model_params)
      opt = torch.optim.Adam(model.parameters(), lr=lr)
      tickers = self._asset_tickers()
      dates = cov_target.index.get_level_values(0).unique()

      model.train()
      for epoch in range(epochs):
        epoch_loss, n_batches = 0.0, 0
        for date in dates:
          if date not in cov_features.index:
              continue
          x = torch.tensor(
              node_arr[cov_features.index.get_loc(date)],
              dtype=torch.float32
          )
          target = torch.tensor(
            cov_target.loc[date]\
                  .reindex(index=tickers, columns=tickers)\
                  .fillna(0.0)\
                  .values,
            dtype=torch.float32
          )
          opt.zero_grad()
          loss = nn.functional.mse_loss(model(x, edge_index), target)
          loss.backward()
          opt.step()
          epoch_loss += loss.item()
          n_batches += 1

        if self.debug and n_batches:
          print(f"[cov] epoch {epoch+1}/{epochs} loss={epoch_loss/n_batches:.6f}")

      self.cov_model = model
      self.edge_index = edge_index

      torch.save(
          {
              "state_dict": model.state_dict(), "params": asdict(cov_model_params),
              "edge_index": edge_index
          },
          os.path.join(self.model_dir, "cov_model.pt")
      )

      return model

  def train(
      self,
      ex_r_data,
      cov_data,
      ex_r_model_params,
      cov_model_params,
      **kwargs
    ):
      ex_r_features, ex_r_target = ex_r_data
      cov_features, cov_target = cov_data
      self.train_ex_r_model(ex_r_model_params, ex_r_features, ex_r_target, **kwargs)
      self.train_cov_model(cov_model_params, cov_features, cov_target, **kwargs)

  def _load_ex_r_model(self):
    if self.ex_r_model is not None:
      return self.ex_r_model

    ckpt = torch.load(os.path.join(self.model_dir, "ex_r_model.pt"))
    params = ExRModelParams(**ckpt["params"])
    model = SpatialTemporalTransformer(params)
    model.load_state_dict(ckpt["state_dict"])

    model.eval()
    self.ex_r_model = model

    return model

  def _load_cov_model(self):
    if self.cov_model is not None:
      return self.cov_model, self.edge_index

    ckpt = torch.load(os.path.join(self.model_dir, "cov_model.pt"))
    params = CovModelParams(**ckpt["params"])
    model = CovarianceGNN(params)
    model.load_state_dict(ckpt["state_dict"])

    model.eval()
    self.cov_model = model
    self.edge_index = ckpt["edge_index"]

    return model, self.edge_index


  def generate_ex_returns(self, ex_r_features, prev_returns, seq_len):
    model = self._load_ex_r_model()
    X, _ = self._stack_ex_r_tensor(ex_r_features, seq_len)

    model.eval()
    with torch.no_grad():
      log_r_pred = model(X[-1:]).squeeze(0).numpy()

    log_r_series = pd.Series(log_r_pred, index=self._asset_tickers())

    return self.convert_to_simple(log_r_series, prev_returns)

  def generate_fwd_cov(self, cov_features):
    model, edge_index = self._load_cov_model()
    node_arr = self._cov_node_features(cov_features)
    x_last = torch.tensor(node_arr[-1], dtype=torch.float32)

    model.eval()
    with torch.no_grad():
      cov_pred = model(x_last, edge_index).numpy()

    tickers = self._asset_tickers()

    return pd.DataFrame(cov_pred, index=tickers, columns=tickers)

  def generate_features(self, data, macro_data, market_returns, window=21):
    ex_r_features, ex_r_target = self.generate_ex_R_features(data, window)
    cov_features, cov_target = self.generate_cov_features(
      data,
      macro_data,
      market_returns,
      window
    )

    return ex_r_features, ex_r_target, cov_features, cov_target

  def optimize(self, ex_returns, fwd_cov, w_prev=None):
    w = self.optimize_portfolio(ex_returns, fwd_cov, w_prev=w_prev)
    w_clean = self.process_w(w)
    return pd.Series(w_clean, index=ex_returns.columns)

  def generate_weights(self, data, macro_data, market_returns, prev_returns,
                        seq_len, w_prev=None, window=21):
    ex_r_features, _, cov_features, _ = self.generate_features(
      data, macro_data,
      market_returns,
      window
    )
    ex_returns = self.generate_ex_returns(ex_r_features, prev_returns, seq_len)
    fwd_cov = self.generate_fwd_cov(cov_features)
    return self.optimize(
      pd.DataFrame([ex_returns]),
      fwd_cov.values, w_prev=w_prev
    )